# Homework 12 Part 1: GPTQ Quantization
SID4 = 5330, SEED = 5330, HP_ID = 2 (HP_ID adds no extra setting in this homework). Run on Google Colab, Tesla T4.

In [1]:
%pip install -q optimum gptqmodel
%pip uninstall -q -y torchaudio torchvision

In [2]:
import os, time, gc, torch, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import model_info
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, GPTQConfig

SEED = 5330
torch.manual_seed(SEED); np.random.seed(SEED)
dev = "cuda"
print(torch.cuda.get_device_name(0))

def size_mb(path):
    return sum(os.path.getsize(os.path.join(path, f)) for f in os.listdir(path)) / 1e6

def free():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

@torch.no_grad()
def latency(fn, n=50):
    for _ in range(5): fn()
    t = []
    for _ in range(n):
        torch.cuda.synchronize(); s = time.perf_counter(); fn(); torch.cuda.synchronize()
        t.append((time.perf_counter() - s) * 1000)
    return f"{np.percentile(t, 50):.1f} / {np.percentile(t, 95):.1f}"

Tesla T4


## Base LLM: load and evaluate

In [3]:
MODEL_ID = "facebook/opt-125m"
REV = model_info(MODEL_ID).sha
print("Base model:", MODEL_ID, "revision:", REV)

tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REV)
wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
test_ids = tok("\n\n".join(wiki["test"]["text"]), return_tensors="pt").input_ids
prompt = tok("The history of natural language processing began", return_tensors="pt").to(dev)

@torch.no_grad()
def measure_llm(model, path, L=1024):
    free()
    nll = [model(test_ids[:, i:i + L].to(dev), labels=test_ids[:, i:i + L].to(dev)).loss.float() for i in range(0, test_ids.size(1) - L, L)]
    ppl = torch.exp(torch.stack(nll).mean()).item()
    lat = latency(lambda: model.generate(**prompt, min_new_tokens=32, max_new_tokens=32, do_sample=False))
    return [round(ppl, 2), lat, round(torch.cuda.max_memory_allocated() / 1e9, 3), round(size_mb(path), 1)]

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REV).to(dev).eval()
base.save_pretrained("llm_base")
rows = {"Base LLM": measure_llm(base, "llm_base")}
print(rows)
del base

Base model: facebook/opt-125m revision: 27dcfa74d334bc871f3234de431e71c6eeba5dd6
{'Base LLM': [31.76, '291.6 / 395.8', 0.805, 250.5]}


## GPTQ: quantize, reload, evaluate

In [4]:
calib = [t for t in wiki["train"]["text"] if len(t) > 1000][:128]
q = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REV, device_map="auto",
                                         quantization_config=GPTQConfig(bits=4, dataset=calib, tokenizer=tok))
q.save_pretrained("llm_gptq")
del q
q = AutoModelForCausalLM.from_pretrained("llm_gptq", device_map="auto").eval()
rows["GPTQ quantized LLM"] = measure_llm(q, "llm_gptq")
del q; free()
print(pd.DataFrame(rows, index=["Wikitext2 perplexity", "p50 / p95 latency (ms)", "Peak memory (GB)", "Size on disk (MB)"]).T.to_string())

INFO  Packing Kernel: Auto-selection: adding candidate `TritonV2Linear`
INFO  Kernel: selected -> `TritonV2Linear`.
INFO  Kernel: Auto-selection: adding candidate `ExllamaV2Linear`
INFO  Kernel: selected -> `ExllamaV2Linear`.
                   Wikitext2 perplexity p50 / p95 latency (ms) Peak memory (GB) Size on disk (MB)
Base LLM                          31.76          291.6 / 395.8            0.805             250.5
GPTQ quantized LLM                34.13          558.9 / 731.8             0.68             125.3


## Findings (Table HW11.1: GPTQ comparison on the same LLM and task)
Base model `facebook/opt-125m`, revision `27dcfa74d334bc871f3234de431e71c6eeba5dd6`, used unchanged for both runs. GPTQ at 4 bits with default settings, calibrated on 128 Wikitext2 train passages. Both models evaluated on the Wikitext2 test split on the same Tesla T4. Latency is generation of 32 tokens from a fixed prompt, 50 timed runs after 5 warmups.

| Configuration | Quality metric (perplexity) | p50 / p95 latency (ms) | Peak memory (GB) | Size on disk (MB) |
|---|---|---|---|---|
| Base LLM | 31.76 | 291.6 / 395.8 | 0.805 | 250.5 |
| GPTQ quantized LLM | 34.13 | 558.9 / 731.8 | 0.680 | 125.3 |

* Size on disk is cut in half and peak GPU memory drops about 16 percent. The ratio is 2x and not 4x because the base checkpoint is already stored in 16 bit floats and the embeddings are not quantized.
* Quality cost is small: perplexity rises by 2.37 (about 7.5 percent).
* Latency got worse, not better. The 4 bit weights are unpacked by a custom kernel on every forward pass, so on a model this small GPTQ buys memory and storage, not speed.